## NA2M : single pass through Stage 1 + FAST screening

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

SEED = 42
set_seed(SEED)

### Load config

In [ ]:
from na2m.utils.config import load_na2m_config

CONFIG_PATH = r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas-scores-two-years_tuned.yaml'

config = load_na2m_config(CONFIG_PATH)
config.top_m = 10  # NA2M-specific, not in tuned yaml

print(config)

### Load and preprocess data

In [ ]:
from na2m.data.data_utils import load_compas, preprocess, split

DATA_PATH = r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\datasets\raw\compas-scores-two-years.csv'

df = load_compas(DATA_PATH)
X, y, feature_meta = preprocess(df)

print(f'Samples: {X.shape[0]}, Features: {X.shape[1]}')
for fm in feature_meta:
    print(f'  {fm.name:25s} type={fm.type}', end='')
    if fm.type == 'cat':
        print(f'  n_levels={fm.n_levels}')
    else:
        print(f'  [{fm.min:.3f}, {fm.max:.3f}]')

### Split and build DataLoaders

In [ ]:
from nam.data.dataset import NAMDataset
from torch.utils.data import DataLoader

X_train, X_val, X_test, y_train, y_val, y_test = split(
    X, y, val_frac=0.15, test_frac=0.15, seed=SEED
)

train_dataset = NAMDataset(X_train, y_train)
val_dataset   = NAMDataset(X_val,   y_val)
pool_dataset  = NAMDataset(
    np.concatenate([X_train, X_val]),
    np.concatenate([y_train, y_val]),
)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=config.batch_size, shuffle=False)
pool_loader  = DataLoader(pool_dataset,  batch_size=config.batch_size, shuffle=False)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Pool: {len(pool_dataset)}')

### Build the NA2M model

In [ ]:
from na2m.models.na2m import NA2M

set_seed(SEED)

model = NA2M(
    num_features=X.shape[1],
    feature_meta=feature_meta,
    num_units=config.num_units,
    hidden_sizes=config.hidden_sizes,
    dropout=config.dropout,
    feature_dropout=config.feature_dropout,
    activation=config.activation,
    inter_units=config.inter_units,
    inter_hidden=config.inter_hidden,
)
print(model)

### Stage 1: train main effects

In [ ]:
from na2m.training.fit_na2m import stage1_main

stage1_main(model, train_loader, val_loader, pool_loader, config)
print('Stage 1 done, main effects trained and centered.')

### Stage 1 prediction score (AUROC on test set)

In [ ]:
from nam.training.metrics import auroc

model.eval()
all_logits, all_targets = [], []

test_dataset_nb = NAMDataset(X_test, y_test)
test_loader_nb  = DataLoader(test_dataset_nb, batch_size=config.batch_size, shuffle=False)

with torch.no_grad():
    for X_batch, y_batch, _ in test_loader_nb:
        logits, _ = model(X_batch)
        all_logits.append(logits)
        all_targets.append(y_batch)

test_auc = auroc(torch.cat(all_logits), torch.cat(all_targets))
print(f'Stage 1 test AUROC: {test_auc:.4f}')

### FAST screening rank all candidate pairs

In [ ]:
from na2m.selection.fast import fast_screen


# Collect the full pool as numpy arrays for the screener
X_pool = np.concatenate([X_train, X_val])
y_pool = np.concatenate([y_train, y_val])

ranked_pairs = fast_screen(model, X_pool, y_pool, task=config.task)

print(f'\nAll {len(ranked_pairs)} pairs ranked by FAST interaction strength (best first):')
for rank, (j, k) in enumerate(ranked_pairs):
    print(f'  {rank+1:2d}. ({feature_meta[j].name}, {feature_meta[k].name})')

top_m = ranked_pairs[:config.top_m]
print(f'\nTop-{config.top_m} selected for block training:')
for j, k in top_m:
    print(f'  ({feature_meta[j].name}, {feature_meta[k].name})')

### Stage 2 — block-train interaction subnets

In [ ]:
from na2m.training.fit_na2m import stage2_select
from na2m.selection.policy import NoGate

# clarity was 0.0 in tuned config; set non-zero so the penalty fires during block-training
config.clarity_regularization = 0.1

stage2_select(model, train_loader, val_loader, pool_loader, config, selection_policy=NoGate())
print("Stage 2 done — interactions block-trained and centered.")
print("Active pairs:", model.active_interaction_pairs())

### Per-subnet contributions after Stage 2

In [ ]:
model.eval()

  # Model structure
print(f"Main subnets:        {len(model.main_nns)}")
print(f"Interaction subnets: {len(model.inter_nns)}")
print(f"Active pairs: {model.active_interaction_pairs()}")

# AUC on test set
all_logits, all_targets = [], []
with torch.no_grad():
    for X_batch, y_batch, _ in test_loader_nb:
        logits, _ = model(X_batch)
        all_logits.append(logits)
        all_targets.append(y_batch)

auc = auroc(torch.cat(all_logits), torch.cat(all_targets))
print(f"\nStage 2 test AUROC: {auc:.4f}  (Stage 1 was 0.7549)")